# The whole project in one pass

*Which models get volatility right? Overconfidence in machine-learning and GARCH forecasts of stock
return variance.* Master dissertation, Catolica Lisbon 2025/26, supervisor Nicolo Bertani.

**This notebook reads; it does not rebuild.** Every artefact below is produced by the package in
`src/thesis/` and rebuilt with `make`. The notebook opens each one in turn so the structure of the
project is visible in one scroll, and runs one model live on one stock so the contract that holds it
all together is not just described but shown. Running every cell takes seconds; rebuilding the
pipeline from scratch takes about half an hour.

> **Kernel.** Select the repo venv (`.venv`, Python 3.14). The kernel registered as "Python 3" on
> this machine is the *system* Python 3.12 and does not have the requirements, so `import thesis`
> will fail there. To register the venv once:
> `.venv\Scripts\python -m ipykernel install --user --name thesis-venv --display-name "Thesis (.venv)"`

## The chain

```
config.yaml ........ frozen design values, read by every module
DECISIONS.md ....... why each value is what it is, and what is still open
        |
raw ----+---> panel.parquet ---> features.parquet ---> forecasts.parquet ---> scores + figures
VOLARE csv          40 x 2,932        24 columns          2.09 million          tables/ figures/
Yahoo prices        stock-days        no look-ahead       forecasts             manuscript/
earnings flags
```

Two settings share that chain: **VOLARE** (40 US stocks, 2015-2026, realised variance observed, so
under-estimation can be tested directly) and **M6** (100 assets, 2022-23, scored against the
competition leaderboard). Everything below is the VOLARE side, which is built; the M6 side is the
piece still to come.

## Where the other notebooks fit

| notebook | what it covers |
|---|---|
| `data_preparation_tutorial.ipynb` | the data stage done by hand, step by step, then checked against the pipeline |
| `decisions_2_3.ipynb` | the evidence behind decisions 2 and 3, with the options laid out |
| `data_overview.ipynb` | exploration: coverage, summary statistics, persistence, the cross-section |
| **this one** | the structure end to end: what each stage produces and where each choice is recorded |

## 1. The frozen design values

Nothing that belongs in `config.yaml` is hard-coded anywhere else. Change a value here, write why in `DECISIONS.md`, re-run the pipeline.

In [1]:
import numpy as np
import pandas as pd

from thesis.paths import (FEATURES, FIGURES, FORECASTS, PANEL, REALISED, ROOT, SCORES, TABLES,
                          load_config)

cfg = load_config()
print("sample       ", cfg["sample"]["start"], "->", cfg["sample"]["end"],
      f'({len(cfg["sample"]["tickers"])} stocks, market proxy {cfg["sample"]["market_proxy"]})')
print("target       ", cfg["target"]["intraday_measure"], "+ overnight^2   |  earnings days ->",
      cfg["target"]["earnings_days"])
print("horizons     ", cfg["horizons"], "trading days")
print("first origin ", cfg["evaluation"]["first_origin"], " refit:", cfg["evaluation"]["origin_frequency"])
print("interval lvls", cfg["evaluation"]["interval_levels"])
print("models       ", list(cfg["models"]))
print("seed         ", cfg["seed"])

sample        2015-01-02 -> 2026-08-31 (40 stocks, market proxy IVV)
target        rv5_ss + overnight^2   |  earnings days -> medrv5
horizons      [1, 20] trading days
first origin  2018-01-02  refit: monthly
interval lvls [0.9, 0.95, 0.99]
models        ['naive', 'garch', 'gjr', 'har', 'har_pooled', 'adavol', 'lightgbm']
seed          20260916


## 2. Raw to panel

`src/thesis/data/build_panel.py` joins the VOLARE realised measures, Yahoo prices (returns,
dividends, splits) and the after-close earnings flags into one balanced stock-day panel.

Two decisions are baked in here and both change what "the truth" means:

- **Decision 2** -- the target must cover the same hours as the return the models forecast, so the
  squared overnight return is added to the intraday measure. Without it a model that is exactly
  right would look like it over-forecasts by about 60 percent.
- **Decision 3** -- on 459 after-close earnings days the announcement jump sits inside VOLARE's
  16:05 window but belongs to the *next* day's return, where it already enters through the overnight
  term. On those days `rv5_ss` is replaced by the jump-robust `medrv5`, so the news is counted once.

In [2]:
p = pd.read_parquet(PANEL)
print(f"panel.parquet: {len(p):,} rows = {p.symbol.nunique()} stocks x {p.date.nunique()} days, "
      f"{p.date.min().date()} -> {p.date.max().date()}")
print("overnight share of the target (median): %.1f%%" % (100 * (p.ret_overnight**2 / p.rv_target).median()))
print("after-close earnings days flagged:", int(p.earnings_afterclose.sum()))

# Decision 3 in one example: NFLX reported after the close on 19 Apr 2022 and fell about 20% between
# 16:00 and 16:05. rv5_ss holds that jump; rv_intraday_used takes medrv5 instead on the flagged day,
# and the move comes back the next morning through ret_overnight. Variances are shown x 1e4.
ex = p[p.symbol == "NFLX"].set_index("date").loc["2022-04-18":"2022-04-21"]
show = (ex[["rv5_ss", "medrv5", "rv_intraday_used", "rv_target"]] * 1e4).round(2)
show["overnight ret"] = ex.ret_overnight.round(4)
show["flagged"] = ex.earnings_afterclose
show

panel.parquet: 117,280 rows = 40 stocks x 2932 days, 2015-01-02 -> 2026-08-31
overnight share of the target (median): 11.7%
after-close earnings days flagged: 459


,rv5_ss,medrv5,rv_intraday_used,rv_target,overnight ret,flagged
date,,,,,,
2022-04-18,4.78,5.09,4.78,4.89,-0.0033,False
2022-04-19,368.08,6.40,6.40,8.31,-0.0138,True
2022-04-20,48.01,39.16,48.01,1286.20,-0.3519,False
2022-04-21,20.33,21.52,20.33,28.03,-0.0277,False


## 3. Panel to features

`src/thesis/features.py` builds the regressors and the targets of decision 9. The rule the whole
project rests on: **every feature uses days up to and including `t`, every target uses days after
`t`.** The cell below re-computes one feature and one target by hand to show the rule holding.

- `y_h1  = log(rv on t+1)`
- `y_h20 = log(mean daily rv over t+1 .. t+20)` -- the *average*, not the sum. The models multiply by
  `h` to get the variance of the cumulative return, which is what the intervals need.

In [3]:
f = pd.read_parquet(FEATURES)
print(f"features.parquet: {f.shape[0]:,} rows x {f.shape[1]} columns")
print(", ".join(f.columns))

g = f[f.symbol == "AAPL"].sort_values("date").reset_index(drop=True)
i = 2000
print(f"\nAAPL, day {g.date.iloc[i].date()} (row {i})")
print("  log_rv_m5 : %+.4f   recomputed from t-4..t : %+.4f"
      % (g.log_rv_m5.iloc[i], np.log(g.rv_target.iloc[i - 4:i + 1]).mean()))
print("  y_h1      : %+.4f   recomputed from t+1    : %+.4f"
      % (g.y_h1.iloc[i], np.log(g.rv_target.iloc[i + 1])))
print("  y_h20     : %+.4f   recomputed from t+1..t+20: %+.4f"
      % (g.y_h20.iloc[i], np.log(g.rv_target.iloc[i + 1:i + 21].mean())))

features.parquet: 117,280 rows x 24 columns
symbol, date, rv_target, log_rv_target, ret, vix, mkt_rv, earnings_afterclose, log_rv_m1, log_rv_m5, log_rv_m22, log_rsn_m1, log_rsp_m1, log_rq_m1, log_mkt_rv_m1, log_mkt_rv_m5, log_mkt_rv_m22, log_vix, ret_m1, neg_ret_m1, jump_share, dow, y_h1, y_h20

AAPL, day 2022-12-12 (row 2000)
  log_rv_m5 : -8.2111   recomputed from t-4..t : -8.2111
  y_h1      : -6.4078   recomputed from t+1    : -6.4078
  y_h20     : -7.7238   recomputed from t+1..t+20: -7.7238


## 4. The forecast-table contract

This is the hinge of the project. Every model is a function

```python
fit_predict(stock_or_panel, windows, horizons, cfg) -> DataFrame
```

returning one row per origin and horizon with the columns in `src/thesis/models/base.py`:

| column | meaning |
|---|---|
| `symbol`, `origin`, `horizon` | the asset, the day the forecast is made, days ahead |
| `fit_origin` | the day the parameters were last estimated (decision 4) |
| `model` | the model name; a log-scale model records its retransformation here |
| `var` | forecast variance of the `h`-day cumulative return |
| `q_0.005 ... q_0.995` | quantiles of the `h`-day return |
| `p_q1 .. p_q5` | M6 quintile probabilities (the M6 setting only -- not written yet) |

Scoring never looks inside a model; it reads only this table. That is what lets GARCH, HAR, a boosted
tree and a reimplemented M6 method be compared without special cases. Below, HAR is fitted live on one
stock for one window.

In [4]:
from thesis.evaluate.rolling import fit_windows
from thesis.models import MODELS, har, is_pooled

windows = fit_windows(f.date, cfg)
print(f"{len(windows)} monthly fit windows, {windows[0][0].date()} ... {windows[-1][0].date()}")
print(f"first window: estimate on data up to {windows[0][0].date()}, "
      f"forecast every trading day until {windows[0][1].date()}")

stock = f[f.symbol == "AAPL"].sort_values("date").reset_index(drop=True)
demo = har.fit_predict(stock, windows[:1], cfg["horizons"], cfg)
print(f"\nHAR on AAPL, first window: {len(demo)} rows, models {sorted(demo.model.unique())}")
demo.head(4)

104 monthly fit windows, 2018-01-02 ... 2026-08-03
first window: estimate on data up to 2018-01-02, forecast every trading day until 2018-02-01

HAR on AAPL, first window: 84 rows, models ['har', 'har_median']


,symbol,origin,fit_origin,horizon,model,var,q_0.005,q_0.025,q_0.05,q_0.25,q_0.5,q_0.75,q_0.95,q_0.975,q_0.995
0,AAPL,2018-01-02,2018-01-02,1,har,0.000106,-0.026518,-0.020178,-0.016934,-0.006944,0.0,0.006944,0.016934,0.020178,0.026518
1,AAPL,2018-01-03,2018-01-02,1,har,0.000089,-0.024336,-0.018517,-0.015540,-0.006372,0.0,0.006372,0.015540,0.018517,0.024336
2,AAPL,2018-01-04,2018-01-02,1,har,0.000084,-0.023666,-0.018008,-0.015112,-0.006197,0.0,0.006197,0.015112,0.018008,0.023666
3,AAPL,2018-01-05,2018-01-02,1,har,0.000074,-0.022171,-0.016870,-0.014158,-0.005806,0.0,0.005806,0.014158,0.016870,0.022171


### The registry

`MODELS` in `src/thesis/models/__init__.py` holds six entries, but the forecast table carries
**twelve** model rows: HAR, pooled HAR and the two LightGBM variants each write a second row for
their retransformation. The plain exponential of a log forecast is the conditional *median* of the
variance; the conditional mean is `exp(mu + s^2/2)`. The gap between them is one of the candidate
causes of machine-learning overconfidence (decision 9), so it is measured in every table rather than
assumed away.

In [5]:
reg = pd.DataFrame([{"registry key": n,
                     "fitted": "pooled, all stocks at once" if is_pooled(n) else "one fit per stock"}
                    for n in MODELS])
display(reg)

fc_models = pd.read_parquet(FORECASTS, columns=["model"]).model.unique()
print(f"{len(MODELS)} registry entries -> {len(fc_models)} model rows in the forecast table:")
print("  " + ", ".join(sorted(fc_models)))

,registry key,fitted
0,naive,one fit per stock
1,garch,one fit per stock
2,gjr,one fit per stock
3,har,one fit per stock
4,har_pooled,"pooled, all stocks at once"
5,adavol,one fit per stock
6,lgbm,"pooled, all stocks at once"


7 registry entries -> 12 model rows in the forecast table:
  adavol, garch, gjr, har, har_median, har_pooled, har_pooled_median, lgbm, lgbm_median, lgbm_ratio, lgbm_ratio_median, naive


## 5. The rolling-origin driver

`src/thesis/evaluate/rolling.py` is the reading of decision 4: parameters are re-estimated on the
first trading day of every month on an expanding window, and between two fit origins the parameters
are held while a forecast is issued **every trading day** with the data up to that day. So the test
set holds about 2,170 forecasts per stock and model, not 104, and the covid window has about 95 days
rather than 5.

The 20-day forecasts of consecutive days overlap, so their errors are correlated and every test on
them uses HAC standard errors (Mincer-Zarnowitz, and Diebold-Mariano in section 8).

In [6]:
fc = pd.read_parquet(FORECASTS, columns=["model", "symbol", "horizon", "origin", "fit_origin"])
print(f"forecasts.parquet: {len(fc):,} rows")
print(f"  {fc.model.nunique()} model rows x {fc.symbol.nunique()} stocks x {fc.horizon.nunique()} "
      f"horizons x {fc.origin.nunique():,} origins")
print(f"  origins {fc.origin.min().date()} -> {fc.origin.max().date()}   "
      f"({fc.fit_origin.nunique()} fit origins)")

truth = pd.read_parquet(REALISED)
print(f"\nrealised.parquet: {len(truth):,} rows -- the h-day cumulative return and the sum of the "
      f"daily realised variances over origin+1 .. origin+h")
truth.head(3)

forecasts.parquet: 2,089,920 rows


  12 model rows x 40 stocks x 2 horizons x 2,177 origins
  origins 2018-01-02 -> 2026-08-31   (104 fit origins)

realised.parquet: 234,560 rows -- the h-day cumulative return and the sum of the daily realised variances over origin+1 .. origin+h


,symbol,origin,horizon,ret_h,rv_h
0,AAPL,2015-01-02,1,-0.028576,0.000316
1,AAPL,2015-01-05,1,0.000094,0.000446
2,AAPL,2015-01-06,1,0.013925,0.000235


## 6. Scores

`src/thesis/evaluate/run_scores.py` writes `manuscript/tables/scores_volare.md`. Accuracy losses
(QLIKE, MSE) compare the variance forecast with the realised variance on the variance scale, never on
the log scale; calibration scores compare the return quantiles with the realised return.

Aggregation follows decision 10: the mean over origins **within a stock** first, then the mean across
the 40 stocks, so a volatile name cannot dominate a scale-dependent loss.

Reading the columns: `qlike_rel_garch` below 1 beats GARCH(1,1); `mz_beta` below 1 is under-reaction
(the forecast moves too little with the truth); `cov95` should be 0.95.

In [7]:
t = pd.read_parquet(SCORES)
cols = ["model", "qlike", "qlike_rel_garch", "log_ratio", "mz_beta", "cov90", "cov95", "cov99"]
for h in cfg["horizons"]:
    print(f"\nhorizon {h}")
    display(t[t.horizon == h][cols].sort_values("qlike").round(3).reset_index(drop=True))


horizon 1


,model,qlike,qlike_rel_garch,log_ratio,mz_beta,cov90,cov95,cov99
0,lgbm,0.340,0.789,-0.183,1.005,0.898,0.938,0.975
1,lgbm_ratio,0.348,0.807,-0.196,0.940,0.898,0.939,0.975
2,har_pooled,0.374,0.869,-0.187,0.984,0.897,0.937,0.974
3,har,0.375,0.870,-0.177,0.970,0.896,0.936,0.973
4,lgbm_median,0.395,0.916,0.016,1.015,0.867,0.917,0.965
5,lgbm_ratio_median,0.402,0.932,0.010,0.953,0.868,0.918,0.965
6,garch,0.431,1.000,-0.325,0.833,0.886,0.943,0.988
7,gjr,0.436,1.011,-0.306,0.816,0.885,0.942,0.987
8,har_pooled_median,0.443,1.028,0.026,0.984,0.865,0.915,0.962
9,har_median,0.447,1.038,0.035,0.982,0.863,0.914,0.962



horizon 20


,model,qlike,qlike_rel_garch,log_ratio,mz_beta,cov90,cov95,cov99
0,har_pooled,0.276,0.922,-0.116,0.960,0.904,0.944,0.979
1,har,0.280,0.935,-0.076,0.933,0.899,0.941,0.978
2,garch,0.300,1.000,-0.129,0.780,0.880,0.938,0.987
3,gjr,0.304,1.017,-0.091,0.764,0.877,0.936,0.986
4,lgbm,0.308,1.028,-0.102,0.928,0.899,0.941,0.977
5,har_pooled_median,0.323,1.078,0.070,0.967,0.877,0.925,0.970
6,lgbm_ratio,0.329,1.098,-0.147,0.734,0.903,0.942,0.978
7,har_median,0.332,1.110,0.103,0.947,0.872,0.922,0.969
8,lgbm_median,0.347,1.160,0.089,1.001,0.871,0.922,0.969
9,lgbm_ratio_median,0.362,1.209,0.047,0.781,0.876,0.924,0.970


**A trap in that table.** `log_ratio` is the mean of `log(realised / forecast)`, and it mostly
measures the *retransformation*, not over- or under-forecasting: the `_median` rows sit near zero by
construction and the mean-corrected rows sit about `-s^2/2` below. A conditional-mean forecast of a
right-skewed variable *should* sit above the typical realisation. The under-estimation claim rests on
`mz_beta` and on coverage, not on this column.

## 7. Results figures

Built by `make figures` (`src/thesis/evaluate/figures.py`), written to `manuscript/figures` as PDF
for the Quarto PDF and PNG for the docx. Decision 13 records the choices inside them.

### The pooling and nonlinearity ladder (decision 12)

LightGBM is pooled *and* nonlinear; HAR is per stock *and* linear, so the gap between them changes two
things at once. The pooled HAR fills the missing cell and splits the gap into a pooling step and a
nonlinearity step.

![ladder](../manuscript/figures/fig_ladder.png)

### QLIKE by regime and by VIX tercile

The covid column is the finding: nearly every model degrades toward or past GARCH there, while the
2022 rate shock is where the trees are strongest. Colour saturates so one outlier (AdaVol in covid,
2.07) does not flatten the rest; the printed numbers are exact.

![regimes](../manuscript/figures/fig_qlike_regime.png)

### Coverage by interval level

Empirical minus nominal coverage. Below zero is an interval that is too narrow. The four `_median`
rows cluster at the left: the retransformation moves coverage about three times more than the choice
of model family does.

![coverage](../manuscript/figures/fig_coverage.png)

### Mincer-Zarnowitz slopes

A slope below one means the forecast does not move enough with the truth -- too low when variance is
high. Every model rejects unbiasedness; naive, GARCH and GJR are the worst offenders.

![mz beta](../manuscript/figures/fig_mz_beta.png)

### Conditional calibration by forecast decile

Every line sits below the 45-degree line by construction (the mean of a log is below the log of a
mean for a right-skewed variable), so the figure is read for its *slope* and for how the gap widens
from the low deciles to the high ones.

![calibration](../manuscript/figures/fig_mz_calibration.png)

### When was the advantage earned?

The Diebold-Mariano table in section 8 gives one number per comparison for the whole sample. This is
the same loss difference left unaccumulated, so a steady climb (a persistent edge) can be told apart
from a cliff (one episode doing all the work). Rising means the first model of the pair is ahead.

At one day the LightGBM advantage is persistent and covid barely dents it. At twenty days the whole
deficit against GARCH is six weeks: level with GARCH for two years, about 34 points lost through
covid, then a slow recovery. Note the axis is the forecast *origin*, so at h = 20 the damage starts
up to twenty trading days before the shaded band.

![loss over time](../manuscript/figures/fig_loss_over_time.png)

### The volatility path itself

The cross-sectional mean of the 40 stocks, realised against three models one day ahead. This is what
a Mincer-Zarnowitz slope below one looks like: at the March 2020 peak realised volatility reaches
about 220 percent a year while every model tops out between 100 and 118, LightGBM lowest of the three.

![volatility path](../manuscript/figures/fig_volatility_path.png)

In [8]:
for png in sorted(FIGURES.glob("*.png")):
    print(f"{png.name:28s} {png.stat().st_size / 1e3:7.0f} kB   (and {png.stem}.pdf)")
print("\nrebuild with:  make figures      (or: .venv\\Scripts\\python -m thesis.evaluate.figures)")

fig_coverage.png                 116 kB   (and fig_coverage.pdf)
fig_ladder.png                   108 kB   (and fig_ladder.pdf)
fig_loss_over_time.png           172 kB   (and fig_loss_over_time.pdf)
fig_mz_beta.png                  119 kB   (and fig_mz_beta.pdf)
fig_mz_calibration.png           188 kB   (and fig_mz_calibration.pdf)
fig_qlike_regime.png             359 kB   (and fig_qlike_regime.pdf)
fig_volatility_path.png          272 kB   (and fig_volatility_path.pdf)

rebuild with:  make figures      (or: .venv\Scripts\python -m thesis.evaluate.figures)


## 8. Does the difference between two models mean anything?

Diebold-Mariano on QLIKE (decision 13). The 20-day forecasts of consecutive origins overlap and the
40 stocks move together, so a test on the pooled 2.1 million rows would be wrong twice over. The loss
difference is averaged across the 40 stocks at each origin first -- which takes the cross-sectional
dependence out of the problem -- and that series of about 2,170 daily differences is tested with
Newey-West standard errors and 20 lags.

In [9]:
from IPython.display import Markdown, display

display(Markdown((TABLES / "dm_volare.md").read_text(encoding="utf-8")))

Diebold-Mariano on QLIKE: the cross-sectional mean loss difference per origin, with
Newey-West standard errors (20 lags). A negative statistic means the row beats
the benchmark; |t| above 1.96 is the 5 percent level. Decision 13.

**Horizon 1** (2,176 origins)

| model | vs GARCH(1,1) | vs HAR pooled |
|---|---|---|
| naive | +6.08 | +11.86 |
| GARCH(1,1) | -- | +11.16 |
| GJR(1,1,1) | +1.47 | +11.78 |
| AdaVol | +1.44 | +4.18 |
| HAR | -10.94 | +0.21 |
| HAR, median | +1.87 | +10.64 |
| HAR pooled | -11.16 | -- |
| HAR pooled, median | +1.47 | +11.32 |
| LightGBM | -11.53 | -6.02 |
| LightGBM, median | -3.59 | +2.47 |
| LightGBM ratio | -9.23 | -3.70 |
| LightGBM ratio, median | -2.64 | +2.92 |

**Horizon 20** (2,157 origins)

| model | vs GARCH(1,1) | vs HAR pooled |
|---|---|---|
| naive | +5.48 | +8.29 |
| GARCH(1,1) | -- | +3.45 |
| GJR(1,1,1) | +1.05 | +3.69 |
| AdaVol | +1.44 | +2.06 |
| HAR | -1.53 | +0.49 |
| HAR, median | +1.04 | +2.09 |
| HAR pooled | -3.45 | -- |
| HAR pooled, median | +0.90 | +2.24 |
| LightGBM | +0.47 | +2.34 |
| LightGBM, median | +1.45 | +2.50 |
| LightGBM ratio | +1.30 | +2.64 |
| LightGBM ratio, median | +1.77 | +2.75 |


**What it says.** At one day LightGBM beats the pooled HAR (-6.02) and both beat GARCH. At twenty days
the pooled HAR beats GARCH (-3.45) while LightGBM *loses* to the pooled HAR (+2.34). The pooling step
itself is not significant at either horizon (+0.21 and +0.49), so the honest reading is that pooling
buys nothing measurable on this panel and the functional form carries both results: nonlinearity helps
at one day and hurts at twenty, where the trees overfit an overlapping target.

## 9. Where every choice is recorded

`DECISIONS.md` has one entry per design choice a reader could question, with its status and the
evidence. The data chapter quotes it. The table below is generated from the file, so it cannot drift.

In [10]:
import re

text = (ROOT / "DECISIONS.md").read_text(encoding="utf-8")
entries = pd.DataFrame([{"#": int(m.group(1)), "decision": m.group(2), "status": m.group(3)}
                        for m in re.finditer(r"^## (\d+)\. (.+?) \((.+?)\)$", text, re.M)])
display(entries.set_index("#"))

,decision,status
#,,
1,Research question and structure,"decided, 7 Oct 2026"
2,Target variance,"decided, 12 Oct 2026"
3,After-close earnings days,"decided, 12 Oct 2026"
4,Forecast origins and horizons,decided
5,Regimes,"decided, revisit after exploration"
6,Which M6 methods to reimplement,open
7,Bloomberg,decided
8,What gets cut first if time runs short,decided
9,"Target variable, by horizon and setting","decided, 12 Oct 2026"


## 10. What is built, what is not

**Built.** The clean panel and features; ten benchmark and machine-learning model rows plus the two
pooled-HAR rows; the rolling-origin driver with a no-leakage test; the scoring run; the five results
figures and the Diebold-Mariano table. 48 unit tests, about a minute to run.

**Not built yet, in order.**

1. **The M6 setting as a module** -- turning a predictive distribution into quintile probabilities,
   carrying the cross-asset correlation, and reusing one "distribution to ranks" step for every
   model. The three validation scripts in `notebooks/` reproduce the AdaVol, Vilar and SebastianR
   submissions, and the RPS and information-ratio functions are written and tested, but nothing joins
   them and no model writes `p_q1..p_q5`. This is the largest remaining piece and the schedule risk.
2. **The results chapter** in `manuscript/index.qmd`, still a skeleton of section stubs.
3. Smaller: MSIS and PIT are in `evaluate/scores.py` but not yet in `run_scores.py`; the regime
   windows of decision 5 leave 325 of the 2,177 test days unassigned, 294 of them the most recent
   fourteen months, which the regime figure shows as an `other` column.

**Open with Nicolo** (asked 15 Oct 2026): which AdaVol to report; whether the correlated AdaVol
variant may be shown; a fourth M6 method or that time on VOLARE; `medrv5` as the main specification;
and now the regime windows above.

## How to rebuild

```
make panel      # data/processed/panel.parquet      40 stocks x 2,932 days
make features   # data/processed/features.parquet
make forecasts  # forecasts.parquet + realised.parquet   (~25 min, LightGBM is most of it)
make scores     # scores*.parquet + manuscript/tables/scores_volare.md
make figures    # manuscript/figures/*.{pdf,png} + manuscript/tables/dm_volare.md
make test       # 48 unit tests, about a minute
```

On Windows without `make`, call the modules from the venv, e.g.
`.venv\Scripts\python -m thesis.evaluate.rolling --models har_pooled` (a subset run replaces only
those model rows in the table).